# (05) Jobs: rate-dist curve

project = ```iP-VAE```, host = ```mach```, device = ```any```

**Motivation**: <br>

Create jobs for all the models that'll go on the rate-distortion curve.

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_IterativeVAE'))
from figures.analysis import plot_convergence
from figures.imgs import plot_weights
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = 'Dropbox/git/_IterativeVAE/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

[
    'copyfits.sh',
    'fit_model.sh',
    'kill_screens.sh',
    'resume_fit.sh',
    'run_sessions.sh',
    'test_tqdm.py',
    'test_tqdm.sh'
]

## Betas (mach)

```<grad|lin>```

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts_mach = collections.defaultdict(list)
tot = 0

In [5]:
# n_seeds = 5
# seeds = range(1, n_seeds + 1)

models = [
    'poisson',
    'gaussian',
    'gaussian-relu',
    'gaussian-softplus',
]

n_total_betas = 8

t_b_combos = {
    8: [4, 8, 10, 12, 16, 20, 24, 32],
    16: [8, 12, 16, 20, 24, 32, 48, 64],
    32: [8, 16, 24, 32, 40, 48, 64, 96],
}
assert all(len(v) == n_total_betas for v in t_b_combos.values())

In [6]:
number = 0

for gpu_i, model_str in enumerate(models):
    model_type, _, latent_act = model_str.partition('-')
    if not len(latent_act):
        latent_act = None
    print(model_type, latent_act)
    for beta_i in range(n_total_betas):
        _selected = [
            (t, v[beta_i]) for t, v
            in t_b_combos.items()
        ]
        number += 1
        if number % 2 == 0:
            _selected = _selected[::-1]

        for seq_len, beta in _selected:
            arg = ' '.join(filter(None, [
                f"--seq_len {seq_len}",
                f"--kl_beta {beta}",
                f"--latent_act '{latent_act}'" if latent_act else '',
                f"--comment t-{seq_len}_b-{beta:0.2g}",
            ]))
            kws = dict(
                device=gpu_i,
                dataset='vH16',
                archi='grad|lin',
                model=model_type,
                args=arg,
                seed=1,
            )
            scripts_mach[gpu_i].append(job_runner_script(**kws))
            tot += 1

poisson None

gaussian None

gaussian relu

gaussian softplus

In [7]:
print(tot)

96

In [8]:
scripts_mach = dict(scripts_mach)
print({k: len(v) for k, v in scripts_mach.items()})

{0: 24, 1: 24, 2: 24, 3: 24}

### Save

In [9]:
n_fits = 6

for gpu_i, scripts in scripts_mach.items():
    scripts_divided = divide_list(scripts, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 4 --comment t-8_b-4 && 
./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 8 --comment t-16_b-8 && 
./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 8 --comment t-32_b-8 && 
./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 16 --comment t-32_b-16

[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 12 --comment t-16_b-12 && 
./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 8 --comment t-8_b-8 && 
./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 10 --comment t-8_b-10 && 
./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 16 --comment t-16_b-16

[PROGRESS] 'mach-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

...

./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 16 --latent_act 'softplus' --comment
t-8_b-16 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 24 --latent_act 'softplus' 
--comment t-16_b-24 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 40 --latent_act 'softplus' 
--comment t-32_b-40 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 48 --latent_act 'softplus' 
--comment t-32_b-48

[PROGRESS] 'mach-cuda3-fit4.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 32 --latent_act 'softplus' 
--comment t-16_b-32 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 20 --latent_act 'softplus' --comment
t-8_b-20 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 24 --latent_act 'softplus' --comment
t-8_b-24 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 48 --latent_act 'softplus' 
--comment t-16_b-48

[PROGRESS] 'mach-cuda3-fit5.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 64 --latent_act 'softplus' 
--comment t-32_b-64 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 96 --latent_act 'softplus' 
--comment t-32_b-96 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 64 --latent_act 'softplus' 
--comment t-16_b-64 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 32 --latent_act 'softplus' --comment
t-8_b-32

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 64 --latent_act 'softplus' 
--comment t-32_b-64 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 96 --latent_act 'softplus' 
--comment t-32_b-96 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 64 --latent_act 'softplus' 
--comment t-16_b-64 && 
./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 32 --latent_act 'softplus' --comment
t-8_b-32

In [11]:
print(scripts_mach)

{
    0: [
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 4 --comment t-8_b-4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 8 --comment t-16_b-8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 8 --comment t-32_b-8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 16 --comment t-32_b-16",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 12 --comment t-16_b-12",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 8 --comment t-8_b-8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 10 --comment t-8_b-10",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 16 --comment t-16_b-16",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 24 --comment t-32_b-24",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 32 --comment t-32_b-32",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 20 --comment t-16_b-20",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 12 --comment t-8_b-12",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 16 --comment t-8_b-16",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 24 --comment t-16_b-24",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 40 --comment t-32_b-40",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 48 --comment t-32_b-48",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 32 --comment t-16_b-32",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 20 --comment t-8_b-20",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 24 --comment t-8_b-24",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 48 --comment t-16_b-48",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 64 --comment t-32_b-64",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 96 --comment t-32_b-96",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 64 --comment t-16_b-64",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 32 --comment t-8_b-32"
    ],
    1: [
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 4 --comment t-8_b-4",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 8 --comment t-16_b-8",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 8 --comment t-32_b-8",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 16 --comment t-32_b-16",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 12 --comment t-16_b-12",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 8 --comment t-8_b-8",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 10 --comment t-8_b-10",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 16 --comment t-16_b-16",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 24 --comment t-32_b-24",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 32 --comment t-32_b-32",
        "./fit_model.sh '1' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 20 --comment t-16_b-20",
        "./fit_model.sh '1' 'vH16' 'gaussian' 

In [12]:
print(scripts_divided)

[
    [
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 4 --latent_act 'softplus' 
--comment t-8_b-4",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 8 --latent_act 'softplus' 
--comment t-16_b-8",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 8 --latent_act 'softplus' 
--comment t-32_b-8",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 16 --latent_act 'softplus'
--comment t-32_b-16"
    ],
    [
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 12 --latent_act 'softplus'
--comment t-16_b-12",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 8 --latent_act 'softplus' 
--comment t-8_b-8",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 10 --latent_act 'softplus' 
--comment t-8_b-10",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 16 --latent_act 'softplus'
--comment t-16_b-16"
    ],
    [
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 24 --latent_act 'softplus'
--comment t-32_b-24",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 32 --latent_act 'softplus'
--comment t-32_b-32",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 20 --latent_act 'softplus'
--comment t-16_b-20",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 12 --latent_act 'softplus' 
--comment t-8_b-12"
    ],
    [
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 16 --latent_act 'softplus' 
--comment t-8_b-16",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 24 --latent_act 'softplus'
--comment t-16_b-24",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 40 --latent_act 'softplus'
--comment t-32_b-40",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 48 --latent_act 'softplus'
--comment t-32_b-48"
    ],
    [
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 32 --latent_act 'softplus'
--comment t-16_b-32",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 20 --latent_act 'softplus' 
--comment t-8_b-20",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 24 --latent_act 'softplus' 
--comment t-8_b-24",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 48 --latent_act 'softplus'
--comment t-16_b-48"
    ],
    [
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 64 --latent_act 'softplus'
--comment t-32_b-64",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 32 --kl_beta 96 --latent_act 'softplus'
--comment t-32_b-96",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 16 --kl_beta 64 --latent_act 'softplus'
--comment t-16_b-64",
        "./fit_model.sh '3' 'vH16' 'gaussian' 'grad|lin' --seed 1 --seq_len 8 --kl_beta 32 --latent_act 'softplus' 
--comment t-8_b-32"
    ]
]